In [ ]:
# # Initial Imports and Variables
import numpy as np
import torch
import torchvision
import time
import matplotlib.pyplot as plt

import matplotlib
import sns
import os

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

load_dir= "./Data/"
results_directory="./Results/"
RANDOM_STATE=2025

class_list=['Floor-Bite', 'Floor-Explore', 'Floor-Poke','Stand-Bite', 'Stand-Eat', 'Stand-Explore', 'Stand-Poke']

In [ ]:
%run Utilities.py
from IPython.core.magic import register_cell_magic
from IPython import get_ipython

@register_cell_magic
def skip(line, cell):
    return

# Cross Validation

In [ ]:
# %% echo skipping
import torch.nn as nn
import torch.optim as optim
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
import torchvision.transforms as transforms
from torchvision.transforms import v2
from sklearn.utils import shuffle
import torch.nn.functional as F
import copy
from itertools import permutations

In [ ]:
def load_all_train_data(data_root, class_list):
    X, y = [], []

    for behavior in class_list:
        data = np.load(
            os.path.join(data_root, f"{behavior}_train.npz")
        )
        X.append(data["clips"])
        y.append(data["labels"])

    return np.concatenate(X), np.concatenate(y)


from sklearn.model_selection import StratifiedShuffleSplit

def repeated_holdout_splits(
    X, y,
    val_size=0.2,
    n_repeats=5,
    random_state=RANDOM_STATE
):
    splitter = StratifiedShuffleSplit(
        n_splits=n_repeats,
        test_size=val_size,
        random_state=random_state
    )

    for repeat_idx, (train_idx, val_idx) in enumerate(
        splitter.split(X, y)
    ):
        yield repeat_idx, train_idx, val_idx


def load_all_test_data(data_root, class_list):
    X, y = [], []

    for behavior in class_list:
        data = np.load(
            os.path.join(data_root, f"{behavior}_test.npz")
        )
        X.append(data["clips"])
        y.append(data["labels"])

    return np.concatenate(X), np.concatenate(y)

import numpy as np

def choose_random(X, y, num_samples, shuffle=True, random_state=RANDOM_STATE):
    """
    Select exactly `num_samples` samples from EACH class.

    Parameters
    ----------
    X : np.ndarray
        Data array of shape (N, ...)

    y : np.ndarray
        Label array of shape (N,)

    num_samples : int
        Number of samples to select per class

    shuffle : bool, default=True
        Whether to shuffle the resulting dataset

    random_state : int or None
        Random seed for reproducibility

    Returns
    -------
    X_new : np.ndarray
        Balanced data array

    y_new : np.ndarray
        Balanced label array
    """

    rng = np.random.default_rng(random_state)

    X_out = []
    y_out = []

    classes = np.unique(y)

    for c in classes:
        idx = np.where(y == c)[0]

        if len(idx) < num_samples:
            raise ValueError(
                f"Class {c} has only {len(idx)} samples, "
                f"but {num_samples} were requested."
            )

        selected = rng.choice(idx, size=num_samples, replace=False)
        X_out.append(X[selected])
        y_out.append(y[selected])

    X_new = np.concatenate(X_out, axis=0)
    y_new = np.concatenate(y_out, axis=0)

    if shuffle:
        perm = rng.permutation(len(y_new))
        X_new = X_new[perm]
        y_new = y_new[perm]

    return X_new, y_new


import torch
from torch.utils.data import Dataset

class ClipTensorDataset(Dataset):
    """
    Dataset for fixed-length video clips.

    Expects:
      X: numpy array of shape (N, T, C, H, W)
      y: numpy array of shape (N,)
    """

    def __init__(self, X, y):
        self.X = torch.from_numpy(X).float() / 255.0
        self.y = torch.from_numpy(y).long()

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


def max_samples_per_class(y):
    return min(np.bincount(y))


In [ ]:
def kfold_cross_validation(net, model_name, data_root):


    train_accuracies, val_accuracies, test_accuracies = [], [], []
    test_metrics, test_confusions, test_normalized_confusions = [], [], []

    n_folds = 5
    
    X_all, y_all = load_all_train_data(  data_root,  class_list)
    X_test_all, y_test_all = load_all_test_data( data_root, class_list)

    assert X_all.shape[1] == 16, f"Expected clip length T=16, got {X_all.shape[1]}"
    assert X_test_all.shape[1] == 16, f"Expected clip length T=16, got {X_test_all.shape[1]}"

    
    max_train = max_samples_per_class(y_all)
    max_test  = max_samples_per_class(y_test_all)
    print("Max train per class:", max_train)
    print("Max test  per class:", max_test)

    # 63,7,313  # to ensure at least 1000, 0.1*1000,5000 frames for train val test, assuming a clip size of 16, if this amount is available, else whatever we can get
    train_samples = min(64, max_train)
    val_samples   = min(7, max_train // 10)
    test_samples  = min(313, max_test)


    print("Before Split:",X_all.shape,y_all.shape,X_test_all.shape,y_test_all.shape)
    
# Evaluate best model ONCE

    
    
    for i, train_idx, val_idx in repeated_holdout_splits(  X_all, y_all, val_size=0.1, n_repeats=n_folds):
        print(f"\n📁 Fold {i+1}/{n_folds}")
        X_train = X_all[train_idx]
        y_train = y_all[train_idx]
        
        X_val = X_all[val_idx]
        y_val = y_all[val_idx]

        X_train, y_train=choose_random(X_train, y_train, train_samples, random_state=RANDOM_STATE+i)
        X_val, y_val=choose_random(X_val, y_val, val_samples, random_state=RANDOM_STATE+i) 
        X_test, y_test=choose_random(X_test_all, y_test_all, test_samples, random_state=RANDOM_STATE+i)
        
        print(
            "Shapes:",
            X_train.shape, y_train.shape,
            X_val.shape, y_val.shape,
            X_test.shape, y_test.shape
        )
        
        train_dataset = ClipTensorDataset(X_train, y_train)
        val_dataset   = ClipTensorDataset(X_val, y_val)
        test_dataset  = ClipTensorDataset(X_test, y_test)
        
        train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
        val_loader   = DataLoader(val_dataset, batch_size=8, shuffle=False)
        test_loader  = DataLoader(test_dataset, batch_size=8, shuffle=False)
        
        dataloaders = {
            "train": train_loader,
            "val": val_loader,
            "test": test_loader
        }
        
        dataset_sizes = {
            "train": len(train_dataset),
            "val": len(val_dataset),
            "test": len(test_dataset)
        }

        # ---- model ----
        model = copy.deepcopy(net).to(device)

        class_weights = compute_class_weight(
            "balanced",
            classes=np.arange(len(class_list)),
            y=y_train
        )
        class_weights = torch.tensor(class_weights).float().to(device)

        criterion = nn.CrossEntropyLoss(weight=class_weights)

        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=1e-4,
            weight_decay=1e-4
        )

        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=50
        )

        model = train_model(
            model,
            criterion,
            optimizer,
            scheduler,
            dataloaders,
            dataset_sizes,
            num_epochs=50,
            early_stopping_patience=20,
            save_dir=results_directory + model_name + f"/fold_{i}"
        )

        train_accuracies.append(evaluate(model, train_loader, "Train")[0][0])
        val_accuracies.append(evaluate(model, val_loader, "Val")[0][0])

        metrics, confusion, norm_conf = evaluate(model, test_loader, "Test")
        test_accuracies.append(metrics[0])
        test_metrics.append(metrics)
        test_confusions.append(confusion)
        test_normalized_confusions.append(norm_conf)

    # ---- aggregate ----
    test_metrics = np.vstack(test_metrics)
    test_confusions = np.dstack(test_confusions)
    test_normalized_confusions = np.dstack(test_normalized_confusions)

    np.savez_compressed(
        results_directory + model_name + "/history.npz",
        test_metrics=test_metrics,
        test_confusions=test_confusions,
        test_normalized_confusions=test_normalized_confusions
    )

    print("\nAverage Test Metrics:")
    print(100 * test_metrics.mean(axis=0))
    print(100 * test_metrics.std(axis=0))


# Do Everything

In [ ]:
from torchvision.models.video import r3d_18
import torch.nn as nn



class VideoModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.backbone = r3d_18(weights="KINETICS400_V1")
        self.backbone.fc = nn.Linear(
            self.backbone.fc.in_features,
            num_classes
        )

    def forward(self, x):
        """
        x: (B, T, C, H, W)
        """
        x = x.permute(0, 2, 1, 3, 4)  # → (B, C, T, H, W)
        return self.backbone(x)


num_classes = len(class_list)   # defined ONCE, here

model = VideoModel(num_classes=num_classes)
model = model.to(device)

x = torch.randn(2, 16, 3, 224, 224).to(device)
y = model(x)
print(y.shape)

model_name="3DResNet18-Correct"
import warnings
warnings.filterwarnings("ignore") # warnings.resetwarnings()	

print(f"\nRunning cross-validation for  model_name: {model_name}")
kfold_cross_validation(net= model, model_name=model_name,data_root="./Data/newfolder/Data_npz/Behavior_Batched/")